# <u> Copulax Examples </u>

## Multivariate Distributions

CopulAX provides a number of multivariate distribution objects, a full list of which can be found <a href=https://github.com/tfm000/copulax/blob/main/copulax/multivariate/README.md> here</a>.
These distribution objects contain standardised methods, covering almost all intended usecases. Inspection of each object also allows the user to see the implemented parameterisation and other details.

### Parameter Specification

All copulAX distribution objects utilise python dictionaries to label and hold parameters.

Each distribution object implements the `example_params` method, allowing the user to quickly and easily get a sense of what the required parameter key-value naming and form.

In [1]:
from copulax.multivariate import mvt_student_t

print(mvt_student_t.example_params())

{'nu': Array(2.5, dtype=float32), 'mu': Array([[0.],
       [0.],
       [0.]], dtype=float32), 'sigma': Array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]], dtype=float32)}


### Density Functions

All multivariate distribution objects have `pdf` and `logpdf` methods for evaluating the (log) probability density function.

In [ ]:
# generating a random sample
import numpy as np
dim = 3
sample = np.random.normal(loc=0, scale=1, size=(100, dim)) * np.random.standard_t(df=5, size=(100, dim))

# calculating the PDF and log-PDF
example_params = mvt_student_t.example_params(dim=dim)
pdf = mvt_student_t.pdf(sample, params=example_params)
logpdf = mvt_student_t.logpdf(sample, params=example_params)
print("pdf:", pdf[:5])
print("logpdf:", logpdf[:5])

pdf: [[0.0715114 ]
 [0.00481453]
 [0.02403291]
 [0.0181657 ]
 [0.00442849]]


### Generating Random Samples

All copulAX distribution objects are capable of generating random samples using the `rvs` method.
As copulAX is JAX based, a key is required for random number generation. 
Random keys can be generated using copulAX's `get_random_key` function, as shown below.


In [3]:
# generating a random key
from copulax import get_random_key
from jax.random import split
key = get_random_key()
key, subkey = split(key)

# generating a random sample
random_sample = mvt_student_t.rvs(key=subkey, params=example_params, size=100)
print("random sample:", random_sample[:5])

random sample: [[-0.37373784  2.0983357   1.1551142 ]
 [ 0.882044   -0.67823696  0.9052108 ]
 [ 0.39547068  0.15891217  0.79588   ]
 [ 1.6291084  -1.6666582   1.7800558 ]
 [-0.21377528 -0.4825019  -0.28252548]]


### Fitting Distributions to Data

All copulAX distributions are capable of fitting parameters to a given set of observations using the `fit` method.

In [4]:
fitted_params = mvt_student_t.fit(sample)
print("fitted parameters:", fitted_params)

fitted parameters: {'nu': Array(2.6957917, dtype=float32), 'mu': Array([[0.06298425],
       [0.20339186],
       [0.09807049]], dtype=float32), 'sigma': Array([[ 0.5967258 , -0.03380008, -0.01219192],
       [-0.03380008,  0.55635643, -0.0784047 ],
       [-0.01219192, -0.07840471,  0.5198266 ]], dtype=float32)}


### Distribution Statistics

The `stats` method computes key statistics (mean, median, mode, covariance, skewness) from the distribution parameters.

In [ ]:
stats = mvt_student_t.stats(params=example_params)
for stat_name, stat_val in stats.items():
    print(f"{stat_name}: {stat_val}")

### Correlation and Covariance Utilities

CopulAX provides utility functions for generating and working with correlation/covariance matrices.

In [ ]:
from copulax.multivariate import corr, cov, random_correlation, random_covariance
from copulax import get_random_key
from jax.random import split
import jax.numpy as jnp

key = get_random_key()

# generate a random correlation matrix
key, subkey = split(key)
R = random_correlation(subkey, dim=dim)
print("Random correlation matrix:\n", R)

# generate a random covariance matrix
key, subkey = split(key)
C = random_covariance(subkey, dim=dim)
print("Random covariance matrix:\n", C)

# compute correlation and covariance from data
data_corr = corr(jnp.array(sample))
data_cov = cov(jnp.array(sample))
print("Data correlation:\n", data_corr)
print("Data covariance:\n", data_cov)

### JIT Compilation

Most evaluation methods are compatible with JAX's `jit` for accelerated computation.

In [ ]:
from jax import jit

# JIT-compiled log-PDF
jit_logpdf = jit(mvt_student_t.logpdf)
jit_result = jit_logpdf(sample, params=example_params)
print("JIT logpdf:", jit_result[:5])

# JIT-compiled sampling
jit_rvs = jit(mvt_student_t.rvs, static_argnums=(0,))
key, subkey = split(key)
jit_sample = jit_rvs(100, example_params, subkey)
print("JIT sample shape:", jit_sample.shape)

### Saving and Loading Fitted Distributions

Fitted distributions can be saved to disk and loaded back in a later session using the `.save()` method and `copulax.load()` function. Files use the `.cpx` format and are cross-platform (Windows, macOS, Linux).

In [ ]:
import copulax

# save the fitted multivariate distribution
fitted_mvt = mvt_student_t._fitted_instance(fitted_params, name="my_mvt_model")
fitted_mvt.save("fitted_mvt_student_t.cpx")

# load it back
loaded = copulax.load("fitted_mvt_student_t.cpx")
print("loaded parameters:", loaded.params)
print("logpdf matches:", jnp.allclose(
    mvt_student_t.logpdf(sample[:5], params=fitted_params),
    loaded.logpdf(sample[:5])
))

# clean up
import os
os.remove("fitted_mvt_student_t.cpx")